# Topic Modeling Pipeline — JMOe Bibliometric Analysis

This notebook implements a complete **BERTopic** pipeline to extract, model, and analyse latent research topics from academic publications collected from the *Journal of Microwaves, Optoelectronics and Electromagnetic Applications* (JMOe, 2010–2025).

It is part of the paper:
> Freitas, H., Araújo, G., Silva, G., Lobato, F., & Jacob Jr., A. (2025). *Trends in Microwaves, Optoelectronics and Electromagnetics: A Bibliometric Analysis of JMOe (2010–2025)*. Journal of Microwaves, Optoelectronics and Electromagnetic Applications.

---

## What this notebook does

| Step | Description |
|------|-------------|
| **1. Setup** | Mount Google Drive, install dependencies, import libraries |
| **2. Data Loading** | Load pre-processed CSV with titles and abstracts |
| **3. Preprocessing** | spaCy-based text cleaning: lowercasing, lemmatisation, stopword removal, custom substitutions |
| **4. Topic Modeling** | BERTopic with `all-mpnet-base-v2` embeddings, HDBSCAN clustering, and outlier reduction |
| **5. LLM Labeling** | Topic labels generated via DeepSeek-LLM and Gemma (HuggingFace) using KeyBERT as fallback |
| **6. Saving** | Persist the trained model and labelled DataFrame |
| **7. Visualisation** | UMAP scatter plots, datamapplot, heatmap, temporal line chart, word clouds, bar charts |

---

## Directory layout expected on Google Drive

```
MyDrive/
└── JMOe_topic_modelling/          ← BASE_PATH
    ├── data/
    │   ├── artigos_pre.csv        ← input: raw articles
    │   ├── df_unlabeled.csv       ← intermediate: preprocessed texts
    │   └── df_unlabeled_topics.csv← intermediate: texts + topic assignments
    ├── models/
    │   └── topic_model_final.pkl  ← saved BERTopic model
    ├── results/
    │   └── topics_model.csv       ← topic info export
    └── figures/                   ← all generated plots
```

Set `BASE_PATH` in **Section 1** below to point to your Drive folder.


# 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Path configuration ──────────────────────────────────────────────────────
# Update this to point to your project folder on Google Drive
BASE_PATH = "/content/drive/MyDrive/JMOe_topic_modelling"

DATA_PATH    = f"{BASE_PATH}/data"
MODELS_PATH  = f"{BASE_PATH}/models"
RESULTS_PATH = f"{BASE_PATH}/results"
FIGURES_PATH = f"{BASE_PATH}/figures"

import os
for p in [DATA_PATH, MODELS_PATH, RESULTS_PATH, FIGURES_PATH]:
    os.makedirs(p, exist_ok=True)

print("Directories ready.")

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -qqq bertopic sentence-transformers nltk transformers accelerate \
             datamapplot kaleido colorcet wordcloud

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import gc
import re
import os
from collections import Counter, defaultdict

import pandas as pd
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, TextGeneration
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer

import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

import datamapplot
import colorcet
from wordcloud import WordCloud
from scipy.cluster import hierarchy as sch

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Imports complete.")

# 2. Data Loading

In [ ]:
df = pd.read_csv(f"{DATA_PATH}/artigos_pre.csv")

# Standardise column names
df = df.rename(columns={"titulo": "title", "ano": "year", "autores": "author"})

print(f"Loaded {len(df)} records. Columns: {list(df.columns)}")
df.head()

In [ ]:
df_unlabeled = df.copy()
df_unlabeled['text'] = (df_unlabeled['title'].fillna('') + " " +
                        df_unlabeled['abstract'].fillna('')).str.strip()

print(f"Missing titles   : {df_unlabeled['title'].isna().sum()}")
print(f"Missing abstracts: {df_unlabeled['abstract'].isna().sum()}")
print(f"Year distribution:")
print(df_unlabeled['year'].value_counts().sort_index())

# 3. Text Preprocessing

In [ ]:
nlp = spacy.load("en_core_web_sm")

CUSTOM_STOPWORDS = {
    "use", "based", "study", "research", "result", "paper", "objective",
    "methods", "method", "material", "resume", "abstract", "information",
    "introduction", "conclusion", "references", "right", "context",
    "finding", "results", "citation", "findings", "good", "literature",
    "methodology", "chapter", "section", "subsection", "cookie", "author",
    "article", "para", "que", "los", "por", "privacidad", "este", "estudio",
}

SUBSTITUTIONS = {
    "datum": "data",
    "ethics": "ethic",
    "ethical": "ethic",
    "customers": "customer",
    "services": "service",
    "studies": "study",
    "development": "develop",
}


def preprocess_text(text: str) -> str:
    """Clean and lemmatise English text for topic modelling."""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"\d+", "", text)               # remove digits
    text = re.sub(r"[^\w\s]", " ", text)          # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()

    doc = nlp(text)
    tokens = [
        SUBSTITUTIONS.get(token.lemma_, token.lemma_)
        for token in doc
        if not token.is_stop
        and token.lemma_ not in CUSTOM_STOPWORDS
        and len(token.lemma_) > 2
    ]
    return " ".join(tokens)

In [ ]:
df_unlabeled['text'] = df_unlabeled['text'].apply(preprocess_text)

# Quick vocabulary check
all_text = ' '.join(df_unlabeled['text'])
top_words = Counter(all_text.split()).most_common(20)
print("Top 20 tokens after preprocessing:")
for word, freq in top_words:
    print(f"  {word}: {freq}")

In [ ]:
df_unlabeled.to_csv(f"{DATA_PATH}/df_unlabeled.csv", index=False)
print(f"Saved preprocessed data → {DATA_PATH}/df_unlabeled.csv")

# 4. Topic Modeling with BERTopic

In [ ]:
# Reload preprocessed data (run this cell if starting from a saved checkpoint)
df_unlabeled = pd.read_csv(f"{DATA_PATH}/df_unlabeled.csv")
print(f"Loaded {len(df_unlabeled)} records.")
df_unlabeled.head()

In [ ]:
# ── HuggingFace login (required to download LLM models) ──────────────────────
from huggingface_hub import login
login()  # enter your HuggingFace token when prompted

## 4.1 LLM Representation Models

Two generative models are loaded for topic labelling:
- **DeepSeek-LLM 7B Chat** — primary labeller
- **Gemma 7B IT** — secondary labeller

KeyBERT is also included as a statistical fallback.

In [ ]:
from transformers import pipeline

LLM_PROMPT = """
Q: I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the information above, provide a short, clear label (max 5 words) that best summarizes the topic.
A:
"""

generator_deepseek = pipeline(
    "text-generation",
    model="deepseek-ai/deepseek-llm-7b-chat",
    device_map="auto",
    torch_dtype="auto",
)

generator_gemma = pipeline(
    "text-generation",
    model="google/gemma-7b-it",
    device_map="auto",
    torch_dtype="auto",
)

representation_model = {
    "KeyBERT": KeyBERTInspired(),
    "LLM_Gemma": TextGeneration(generator_gemma, prompt=LLM_PROMPT),
    "LLM_DeepSeek": TextGeneration(generator_deepseek, prompt=LLM_PROMPT),
}

gc.collect()

## 4.2 Model Configuration

In [ ]:
# ── Embedding model ───────────────────────────────────────────────────────────
embedding_model = SentenceTransformer("all-mpnet-base-v2")

# ── Sub-models ────────────────────────────────────────────────────────────────
cluster_model    = HDBSCAN(min_cluster_size=2, min_samples=2, prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))
ctfidf_model     = ClassTfidfTransformer(bm25_weighting=True)
umap_model       = UMAP(n_neighbors=10, min_dist=0.0, n_components=5, metric='cosine')

# ── BERTopic ──────────────────────────────────────────────────────────────────
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    hdbscan_model=cluster_model,
    representation_model=representation_model,
    calculate_probabilities=True,
    verbose=True,
    nr_topics=8,
)

gc.collect()
print("Model configured.")

## 4.3 Fit & Transform

In [ ]:
docs = (
    df_unlabeled['text']
    .fillna('')
    .astype(str)
)
docs = [d for d in docs if d.strip()]

topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()

In [ ]:
# ── Reduce outliers and update model ─────────────────────────────────────────
topics = topic_model.reduce_outliers(docs, topics, probabilities=probs, strategy="probabilities")
topic_model.update_topics(docs, topics=topics)
topic_model.get_topic_info()

## 4.4 Hierarchical Topics

In [ ]:
linkage_function = lambda x: sch.linkage(x, 'single', optimal_ordering=True)
hierarchical_topics = topic_model.hierarchical_topics(docs, linkage_function=linkage_function)

topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
tree = topic_model.get_topic_tree(hierarchical_topics)
print(tree)

# 5. Save Model and Labelled Data

In [ ]:
topic_model.save(f"{MODELS_PATH}/topic_model_final.pkl")
topic_model.get_topic_info().to_csv(f"{RESULTS_PATH}/topics_model.csv", index=False)
print(f"Model saved  → {MODELS_PATH}/topic_model_final.pkl")
print(f"Topics saved → {RESULTS_PATH}/topics_model.csv")

In [ ]:
# ── Assign topic labels to documents ─────────────────────────────────────────
topic_info  = topic_model.get_topic_info()
topic_names = topic_info.set_index("Topic")["Name"].to_dict()          # default
topic_llm   = topic_info.set_index("Topic")["LLM_DeepSeek"].to_dict()  # LLM labels
topic_kbert = topic_info.set_index("Topic")["KeyBERT"].to_dict()        # KeyBERT labels

mask = df_unlabeled['text'].fillna('').str.strip().str.len() > 5
df_unlabeled = df_unlabeled.dropna(subset=["text"])

df_unlabeled["Topic"]         = topics
df_unlabeled["Topic_Default"] = df_unlabeled["Topic"].map(topic_names)
df_unlabeled["Probability"]   = probs.max(axis=1)

# Clean LLM labels
topics_llm_raw = {k: v for k, v in topic_model.get_topics(full=True)["LLM_DeepSeek"].items() if k != -1}
llm_labels = {
    topic: re.sub(r'\W+', ' ', labels[0][0].split("\n")[0].replace('"', '')).strip()
    if labels and isinstance(labels[0], tuple) else "Unlabelled"
    for topic, labels in topics_llm_raw.items()
}

topic_labels_cleaned = {}
for topic_id, raw_label in llm_labels.items():
    clean = re.sub(r'\W+', ' ', str(raw_label)).strip().strip("'")
    topic_labels_cleaned[topic_id] = clean

df_unlabeled["Topic_Label"]  = df_unlabeled["Topic"].map(topic_labels_cleaned)
df_unlabeled["Topic_KeyBERT"] = df_unlabeled["Topic"].map(
    {t: ' | '.join(list(zip(*v))[0][:3]) for t, v in topic_model.topic_aspects_['KeyBERT'].items()}
)

df_unlabeled.to_csv(f"{DATA_PATH}/df_unlabeled_topics.csv", index=False)
print(f"Labelled data saved → {DATA_PATH}/df_unlabeled_topics.csv")
df_unlabeled[['Topic', 'Topic_Default', 'Topic_Label']].drop_duplicates().sort_values('Topic')

# 6. Visualisation

In [ ]:
# Reload from checkpoint (optional)
df_unlabeled = pd.read_csv(f"{DATA_PATH}/df_unlabeled_topics.csv")
print(f"Loaded {len(df_unlabeled)} records.")

## 6.1 Embeddings & UMAP Reduction

In [ ]:
docs = df_unlabeled['text'].fillna('').astype(str).tolist()
embeddings = embedding_model.encode(docs, show_progress_bar=True)

reduced_embeddings = UMAP(
    n_components=2, n_neighbors=15, min_dist=0.0,
    metric="cosine", random_state=42
).fit_transform(embeddings)

# Build label list aligned with documents
topics_list = df_unlabeled["Topic"].tolist()
all_labels  = [llm_labels.get(t, "Unlabelled") for t in topics_list]

print(f"Embeddings shape: {embeddings.shape}")

## 6.2 UMAP Scatter (Plotly)

In [ ]:
plot_df = pd.DataFrame({
    "x": reduced_embeddings[:, 0],
    "y": reduced_embeddings[:, 1],
    "Topic_Label": all_labels
})

fig = px.scatter(
    plot_df, x="x", y="y", color="Topic_Label",
    title="Latent Topic Space (BERTopic + UMAP)",
    labels={"Topic_Label": "Topic"},
    hover_data=["Topic_Label"],
    width=1000, height=800
)
fig.update_layout(
    title_font_size=24, title_x=0.5,
    legend_title_font_size=16, legend_font_size=14,
    font=dict(size=14),
    plot_bgcolor='rgba(240,240,240,0.95)',
    paper_bgcolor='rgba(240,240,240,0.95)',
)
fig.write_html(f"{FIGURES_PATH}/latentspace.html")
fig.show()

## 6.3 Datamapplot

See [BERTopic docs](https://maartengr.github.io/BERTopic/api/plotting/document_datamap.html).

In [ ]:
plt.rcParams['savefig.bbox'] = 'tight'

fig, ax = datamapplot.create_plot(
    reduced_embeddings,
    all_labels,
    label_font_size=11,
    title="Latent Topic Space via BERTopic and UMAP",
    sub_title="Topic labels generated via DeepSeek-LLM",
    label_wrap_width=27,
    use_medoids=True,
    cmap=colorcet.cm.CET_C2,
)
fig.savefig(f"{FIGURES_PATH}/plot_latent_topic.png", bbox_inches="tight", dpi=300)
plt.show()

## 6.4 Temporal Analysis

In [ ]:
topic_year_counts = (
    df_unlabeled
    .groupby(['year', 'Topic_Label'])
    .size()
    .reset_index(name='Count')
)
topic_year_counts.head()

In [ ]:
# ── Heatmap ──────────────────────────────────────────────────────────────────
pivot = topic_year_counts.pivot_table(
    index='Topic_Label', columns='year', values='Count', fill_value=0
)
plt.figure(figsize=(14, 8))
sns.heatmap(pivot, cmap='YlGnBu', annot=True, fmt='g')
plt.title('Number of Documents per Topic and Year')
plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}/topic_year_heatmap.png', dpi=300)
plt.show()

In [ ]:
# ── Line chart (temporal evolution) ─────────────────────────────────────────
fig = px.line(
    topic_year_counts,
    x='year', y='Count', color='Topic_Label',
    markers=True,
    title='Evolution of Research Topics (2010–2025)',
    labels={'Count': 'Number of Documents', 'year': 'Year', 'Topic_Label': 'Topic'},
    line_shape='linear'
)
fig.update_traces(line=dict(width=2), marker=dict(size=8))
fig.update_layout(
    template='simple_white',
    width=1400, height=700,
    font=dict(size=16),
    legend=dict(title='', font=dict(size=14), orientation='v',
                y=1, x=1.02, xanchor='left', yanchor='top'),
    margin=dict(l=60, r=200, t=60, b=60),
    xaxis=dict(tickmode='linear', tick0=2010, dtick=1,
               title='', showgrid=True),
    yaxis=dict(title='Number of Documents', gridcolor='lightgray')
)
fig.write_html(f'{FIGURES_PATH}/topic_year_lines.html')
fig.show()

In [ ]:
# ── Yearly topic overview (text summary) ─────────────────────────────────────
for year in sorted(df_unlabeled['year'].unique()):
    year_data = topic_year_counts[topic_year_counts['year'] == year]
    if not year_data.empty:
        print(f"\nYear: {year} — {len(year_data)} topics")
        for _, row in year_data.sort_values('Count', ascending=False).iterrows():
            print(f"  • {row['Topic_Label']}: {row['Count']} documents")

## 6.5 BERTopic Built-in Visualisations

In [ ]:
# ── Bar chart (top keywords per topic) ───────────────────────────────────────
keybert_labels = {
    topic: ' | '.join(list(zip(*values))[0][:3])
    for topic, values in topic_model.topic_aspects_['KeyBERT'].items()
}
topic_model.set_topic_labels(keybert_labels)
topic_model.visualize_barchart(top_n_topics=20, n_words=10, height=300)

In [ ]:
# ── Word clouds per topic ─────────────────────────────────────────────────────
topics_dict = topic_model.get_topics()
top_n = min(7, len(topics_dict))
cols  = 2
rows  = (top_n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 4))

for i, topic_id in enumerate(range(top_n)):
    row = i // cols
    col = i % cols
    ax  = axes[row, col] if rows > 1 else axes[col]

    if topic_id in topics_dict:
        words = dict(topics_dict[topic_id])
        wc = WordCloud(width=400, height=200, background_color='white').generate_from_frequencies(words)
        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(f"Topic {topic_id}: {keybert_labels.get(topic_id, '')}", fontsize=11)
        ax.axis("off")

if top_n % cols != 0:
    axes[-1, -1].axis("off")

plt.tight_layout()
plt.savefig(f"{FIGURES_PATH}/wordclouds.png", dpi=300, bbox_inches='tight')
plt.show()